# Бинарная классификация табличных данных - прогнозирование сердечных заболеваний

## Технические предварительные сведения
В ходе курса мы на первом этапе загрузим следующие пакеты.

In [ ]:
import pandas as pd
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
keras.utils.set_random_seed(42)

## Загружаем данные

К счастью, набор данных в формате CSV доступен онлайн (автор: [Франсуа Шолле](https://twitter.com/fchollet)) и мы можем загрузить его в Pandas dataframe с помощью очень полезной функции `pd.read_csv`.

In [ ]:
df = pd.read_csv("http://storage.googleapis.com/download.tensorflow.org/data/heart.csv")

In [ ]:
df.shape

In [ ]:
df.head(3)

In [ ]:
df.target.value_counts(normalize=True, dropna=False)

## Предварительная обработка

Этот набор данных содержит как категориальные, так и числовые переменные.

Для последующей обработки будет удобно объединить эти группы переменных в два списка.

In [ ]:
categorical_variables = ['sex', 'cp', 'fbs', 'restecg','exang', 'ca', 'thal']
numerics = ['age', 'trestbps','chol', 'thalach', 'oldpeak', 'slope']

In [ ]:

df1 = pd.get_dummies(df, columns = categorical_variables, dtype=int)

In [ ]:
df1.head(3)

In [ ]:
test_df = df1.sample(frac=0.2, random_state=42)
train_df = df1.drop(test_df.index)

In [ ]:
train_df.shape

In [ ]:
test_df.shape

In [ ]:
means = train_df[numerics].mean()
sd = train_df[numerics].std()

In [ ]:
means

In [ ]:
train_df[numerics]= (train_df[numerics] - means)/sd
test_df[numerics]= (test_df[numerics] - means)/sd

In [ ]:
train_df.head()

In [ ]:
train = train_df.to_numpy()
test = test_df.to_numpy()

In [ ]:
train_X = np.delete(train, 6, axis=1)
test_X = np.delete(test, 6, axis=1)

In [ ]:
train_X.shape, test_X.shape

In [ ]:
train_y = train[:, 6]
test_y = test[:, 6]

In [ ]:
train_y.shape, test_y.shape

In [ ]:
num_columns = train_X.shape[1]

# define the input layer
input = keras.Input(shape=(num_columns,))

# feed the input vector to the hidden layer
# optional: you can give names to each layer to help
# keep track. This doesn't affect the training etc.

h = keras.layers.Dense(16, activation="relu", name="Hidden")(input)

# feed the output of the hidden layer to the output layer
output = keras.layers.Dense(1, activation="sigmoid", name="Output")(h)

# tell Keras that this (input,output) pair is your model
model = keras.Model(input, output)

In [ ]:
model.summary()

In [ ]:
(29 + 1)* 16 + (16 + 1) * 1

In [ ]:
keras.utils.plot_model(model, show_shapes=True)

### Установка параметров оптимизации

In [ ]:
model.compile(optimizer="adam",
              loss="binary_crossentropy",
              metrics=["accuracy"])

## Обучение модели

Для начала обучения нам необходимо определиться с тремя вещами:
* *Размер пакета*. На лекции мы называли его *мини-пакетом*, но, следуя общепринятой практике, с этого момента будем просто называть его *пакетом*. 32 — хорошее значение по умолчанию для размера пакета.

* Количество *эпох*, т.е. сколько проходов по обучающим данным. Обычно 20-30 эпох — хорошая отправная точка, но поскольку этот набор данных очень мал, каждая эпоха будет очень быстрой, поэтому давайте запустим обучение на 300 эпох. Это позволит нам увидеть, происходит ли переобучение.
*  Хотим ли мы использовать набор данных для валидации. Это будет полезно для обнаружения переобучения и регуляризации с помощью ранней остановки, поэтому мы попросим Keras автоматически использовать 20% точек данных в качестве набора данных для валидации.

In [ ]:
history = model.fit(train_X,              # the array with the input X columns
                    train_y,              # the array with the output y column
                    epochs=300,           # number of epochs to run
                    batch_size=32,        # number of samples (ie data points) per batch
                    verbose=1,            # verbosity during training
                    validation_split=0.2) # use 20% of the data for validation

In [ ]:
history_dict = history.history
history_dict.keys()

In [ ]:
# https://github.com/fchollet/deep-learning-with-python-notebooks/

loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, "bo", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
# https://github.com/fchollet/deep-learning-with-python-notebooks/

plt.clf()
acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
plt.plot(epochs, acc, "bo", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## Оценка модели

Давайте посмотрим, **насколько хорошо модель работает на тестовом наборе данных**.

Функция `model.evaluate` очень удобна для расчета производительности вашей модели на любом наборе данных.

In [ ]:
model.evaluate(test_X, test_y)

## Прогнозирование новых данных (т.е. *вывод*) с помощью модели

В последующих выпусках мы рассмотрим, как сохранить модель Keras и использовать её для прогнозирования. Но мы хотим отметить, что простой подход, который мы использовали в этом выпуске, имеет существенный недостаток.

Мы выполняли предварительную обработку — one-hot кодирование и нормализацию — *вне* модели. Это означает, что нам нужно помнить, какую предварительную обработку мы выполнили, и переносить эту информацию (например, среднее значение и дисперсию каждой переменной) вместе с моделью, чтобы правильно использовать её в будущем.

Очень элегантный способ избежать этой проблемы — использовать [слои предварительной обработки Keras](https://keras.io/guides/preprocessing_layers/). В целях экономии времени мы не будем рассматривать это в данном примере, но рекомендуем вам ознакомиться с этим примером [colab](https://colab.research.google.com/github/keras-team/keras-io/blob/master/examples/structured_data/ipynb/structured_data_classification_from_scratch.ipynb), в котором показано, как решить нашу задачу прогнозирования сердечных заболеваний, используя слои предварительной обработки.